# 0. CONFIG

In [ ]:
import os
import math
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import pandapower as pp
import simbench as sb
from datetime import datetime, timedelta

# Config
RANDOM_SEED = 42
TIME_RESOLUTION_MINUTES = 15
STEPS_PER_DAY = int(24 * 60 / TIME_RESOLUTION_MINUTES)
DATA_DIR = Path("data")
INPUT_PATH = DATA_DIR / "ev_profiles.csv"
OUTPUT_PATH = DATA_DIR / "ev_optimized_profiles.csv"
HC_OUTPUT_PATH = DATA_DIR / "hosting_capacity_results.csv"

np.random.seed(RANDOM_SEED)

# TOU Price Forecast
price_forecast = np.ones(STEPS_PER_DAY) * 0.10
price_forecast[64:84] = 0.30

# GA Config
GA_POP_SIZE = 50
GA_GENERATIONS = 50
MUTATION_RATE = 0.10
ELITISM_RATIO = 0.10
TOURNAMENT_SIZE = 3
WEIGHT_PEAK = 100.0
WEIGHT_COST = 1.0
WEIGHT_SUBNET = 50.0  # Location-aware penalty weight

# Hosting Capacity Config
RUN_HOSTING_CAPACITY = True
PENETRATION_SWEEP = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
# Fast HC evaluation subset: Every 4th step + evening peak 68-88
HC_STEPS = list(set(list(range(0, 96, 4)) + list(range(68, 89))))
HC_STEPS.sort()

# 1. STAGE B: GRID BASELINE
### 1.1 Load Data and Network

In [ ]:
if INPUT_PATH.exists():
    ev_df = pd.read_csv(INPUT_PATH)
    print(f"Loaded {len(ev_df)} EV profiles.")
else:
    print(f"Error: {INPUT_PATH} not found.")
    ev_df = pd.DataFrame()

SIMBENCH_CODE = "1-MVLV-urban-all-0-sw"
print(f"Loading {SIMBENCH_CODE} from SimBench...")
net = sb.get_simbench_net(SIMBENCH_CODE)

for k in net.profiles.keys():
    net.profiles[k] = net.profiles[k].iloc[:STEPS_PER_DAY]
    
load_abs = sb.get_absolute_profiles_from_relative_profiles(net, element="load", multiplying_column="p_mw", profile_suffix="_pload", time_as_index=True)
renew_abs = sb.get_absolute_profiles_from_relative_profiles(net, element="sgen", multiplying_column="p_mw", time_as_index=True)
print("Background load profiles successfully extracted.")

# Build bus-to-subnet dictionary for location-aware penalty
bus_names = net.bus['name'].to_dict()
bus_to_subnet = {}
for b_idx, b_name in bus_names.items():
    if isinstance(b_name, str) and " " in b_name:
        bus_to_subnet[b_idx] = b_name.split(" ")[0]
    else:
        bus_to_subnet[b_idx] = "unknown"

### 1.2 Select EV Buses & 1.3 Assign EVs

In [ ]:
lv_loads = net.load[net.load['bus'].isin(net.bus[net.bus['vn_kv'] == 0.4].index)].copy()
res_buses = lv_loads[lv_loads['profile'].str.contains("H0", na=False)]['bus'].unique()
work_buses = lv_loads[lv_loads['profile'].str.contains("G|L", na=False)]['bus'].unique()
if len(res_buses) == 0: res_buses = lv_loads['bus'].unique()
if len(work_buses) == 0: work_buses = lv_loads['bus'].unique()
public_buses = np.random.choice(lv_loads['bus'].unique(), size=min(10, len(lv_loads['bus'].unique())), replace=False)

def get_candidate_buses(ev_type):
    if ev_type == 'residential': return res_buses
    if ev_type == 'workplace': return work_buses
    return public_buses

if not ev_df.empty:
    MAX_EVS = len(lv_loads)
    print(f"Network capacity (EXTREME = 100%): {MAX_EVS} EVs")
    
    def generate_active_evs(penetration):
        target_evs = int(MAX_EVS * penetration)
        if len(ev_df) > 0:
            active = ev_df.sample(n=target_evs, replace=True, random_state=RANDOM_SEED).copy()
        else:
            active = pd.DataFrame()
            
        active = active.reset_index(drop=True)
        buses = []
        for t in active['ev_type']:
            buses.append(np.random.choice(get_candidate_buses(t)))
        active['assigned_bus'] = buses
        return active

### 1.4 AC Power Flow Engine
Evaluates power flow and records unique violating assets.

In [ ]:
def run_ac_power_flow(net, active_evs_df, load_abs, renew_abs, steps_to_run):
    results = []
    orig_load_p = net.load['p_mw'].copy()
    orig_sgen_p = net.sgen['p_mw'].copy()
    
    for step in steps_to_run:
        net.load['p_mw'] = load_abs.iloc[step].to_numpy()
        net.sgen['p_mw'] = renew_abs.iloc[step].to_numpy()
        
        if active_evs_df is not None and not active_evs_df.empty:
            ev_loads = active_evs_df[['assigned_bus', f'step_{step}']].copy()
            ev_loads['p_mw'] = ev_loads[f'step_{step}'] / 1000.0
            agg_ev_load = ev_loads.groupby('assigned_bus')['p_mw'].sum()
            for bus, p_mw in agg_ev_load.items():
                load_idx = net.load[net.load['bus'] == bus].index
                if len(load_idx) > 0:
                    net.load.at[load_idx[0], 'p_mw'] += p_mw
                    
        try:
            pp.runpp(net, algorithm="nr", init="auto")
        except pp.powerflow.LoadflowNotConverged:
            continue
            
        v_min = net.res_bus['vm_pu'].min()
        v_violations = ((net.res_bus['vm_pu'] < 0.95) | (net.res_bus['vm_pu'] > 1.05)).sum()
        line_load = net.res_line['loading_percent'].max()
        line_violations = (net.res_line['loading_percent'] > 100).sum()
        trafo_load = net.res_trafo['loading_percent'].max()
        trafo_violations = (net.res_trafo['loading_percent'] > 100).sum()
        
        v_viol_buses = net.bus.index[(net.res_bus['vm_pu'] < 0.95) | (net.res_bus['vm_pu'] > 1.05)].tolist()
        l_viol_lines = net.line.index[net.res_line['loading_percent'] > 100].tolist()
        t_viol_trafos = net.trafo.index[net.res_trafo['loading_percent'] > 100].tolist()
        worst_bus = net.res_bus['vm_pu'].idxmin()
        worst_line = net.res_line['loading_percent'].idxmax()
        
        results.append({
            'step': step, 'v_min': v_min, 'line_load_max': line_load, 'trafo_load_max': trafo_load,
            'losses_mw': net.res_line['pl_mw'].sum() + net.res_trafo['pl_mw'].sum(),
            'ext_grid_mw': net.res_ext_grid['p_mw'].sum(),
            'v_violations': v_violations, 'loading_violations': line_violations + trafo_violations,
            'v_viol_buses': v_viol_buses, 'l_viol_lines': l_viol_lines, 't_viol_trafos': t_viol_trafos,
            'worst_bus': net.bus.at[worst_bus, 'name'] if pd.notna(worst_bus) else None,
            'worst_line': net.line.at[worst_line, 'name'] if pd.notna(worst_line) else None
        })
        
    net.load['p_mw'] = orig_load_p
    net.sgen['p_mw'] = orig_sgen_p
    return pd.DataFrame(results)

# 2. STAGE C: GENETIC ALGORITHM
Location-aware Fitness Function & Exact Energy Preservation

In [ ]:
def get_ev_params(active_evs):
    params = []
    for idx, row in active_evs.iterrows():
        arr = int(row['arrival_step'])
        dep = int(row['departure_step'])
        energy = float(row['energy_kwh'])
        power = float(row['max_power_kw'])
        dur = dep - arr if dep > arr else (dep - arr + STEPS_PER_DAY)
        
        energy_per_step = power * (TIME_RESOLUTION_MINUTES / 60.0)
        needed_full = int(energy // energy_per_step)
        needed_partial = (energy % energy_per_step) / (TIME_RESOLUTION_MINUTES / 60.0)
        
        total_steps_needed = needed_full + (1 if needed_partial > 0 else 0)
        total_steps_needed = min(total_steps_needed, dur)
        max_delay = max(0, dur - total_steps_needed)
        
        params.append({
            'index': idx, 'arr': arr, 'needed_full': needed_full, 
            'needed_partial': needed_partial, 'power_mw': power / 1000.0,
            'max_delay': max_delay, 'bus': row['assigned_bus'], 'subnet': bus_to_subnet.get(row['assigned_bus'], 'unknown')
        })
    return params

def run_ga(active_evs, print_convergence=False):
    params = get_ev_params(active_evs)
    num_evs = len(params)
    total_base_load = load_abs.iloc[:STEPS_PER_DAY].sum(axis=1).values
    
    def evaluate_population(population):
        pop_size = population.shape[0]
        fitness = np.zeros(pop_size)
        
        for i in range(pop_size):
            delays = population[i]
            agg_load = total_base_load.copy()
            cost = 0.0
            sub_peaks = {}
            
            for j, ev in enumerate(params):
                start = (ev['arr'] + delays[j]) % STEPS_PER_DAY
                sub = ev['subnet']
                if sub not in sub_peaks: sub_peaks[sub] = np.zeros(STEPS_PER_DAY)
                
                for s in range(ev['needed_full']):
                    step = (start + s) % STEPS_PER_DAY
                    agg_load[step] += ev['power_mw']
                    cost += ev['power_mw'] * price_forecast[step]
                    sub_peaks[sub][step] += ev['power_mw']
                if ev['needed_partial'] > 0:
                    step = (start + ev['needed_full']) % STEPS_PER_DAY
                    agg_load[step] += (ev['needed_partial'] / 1000.0)
                    cost += (ev['needed_partial'] / 1000.0) * price_forecast[step]
                    sub_peaks[sub][step] += (ev['needed_partial'] / 1000.0)
            
            local_penalty = sum(np.max(arr) for arr in sub_peaks.values())
            fitness[i] = np.max(agg_load) * WEIGHT_PEAK + cost * WEIGHT_COST + local_penalty * WEIGHT_SUBNET
        return fitness

    max_delays = np.array([p['max_delay'] for p in params])
    population = np.zeros((GA_POP_SIZE, num_evs), dtype=int)
    for i in range(GA_POP_SIZE):
        if i == 0: population[i] = np.zeros(num_evs, dtype=int)
        else: population[i] = np.array([np.random.randint(0, md + 1) for md in max_delays])

    fitness_history = []
    base_peak_val = np.max(total_base_load)
    base_peak_step = np.argmax(total_base_load)
    
    for gen in range(GA_GENERATIONS):
        fitness = evaluate_population(population)
        sort_idx = np.argsort(fitness)
        population = population[sort_idx]
        fitness = fitness[sort_idx]
        best_fit = fitness[0]
        fitness_history.append(best_fit)
        
        next_gen = np.zeros_like(population)
        num_elites = max(1, int(GA_POP_SIZE * ELITISM_RATIO))
        next_gen[:num_elites] = population[:num_elites]
        
        for i in range(num_elites, GA_POP_SIZE):
            t1 = np.random.choice(GA_POP_SIZE, TOURNAMENT_SIZE, replace=False)
            t2 = np.random.choice(GA_POP_SIZE, TOURNAMENT_SIZE, replace=False)
            p1 = population[t1[np.argmin(fitness[t1])]]
            p2 = population[t2[np.argmin(fitness[t2])]]
            
            mask = np.random.rand(num_evs) < 0.5
            child = np.where(mask, p1, p2)
            mut_mask = np.random.rand(num_evs) < MUTATION_RATE
            random_genes = np.array([np.random.randint(0, md + 1) for md in max_delays])
            child = np.where(mut_mask, random_genes, child)
            next_gen[i] = child
            
        population = next_gen

    best_chromosome = population[0]
    best_active_evs = active_evs.copy()
    for j, ev in enumerate(params):
        idx = ev['index']
        delay = best_chromosome[j]
        start = (ev['arr'] + delay) % STEPS_PER_DAY
        
        for s in range(STEPS_PER_DAY):
            best_active_evs.at[idx, f'step_{s}'] = 0.0
            
        for s in range(ev['needed_full']):
            step = (start + s) % STEPS_PER_DAY
            best_active_evs.at[idx, f'step_{step}'] = ev['power_mw'] * 1000.0
        if ev['needed_partial'] > 0:
            step = (start + ev['needed_full']) % STEPS_PER_DAY
            best_active_evs.at[idx, f'step_{step}'] = ev['needed_partial']
            
    return best_active_evs

# 3. HOSTING CAPACITY LOOP (STAGE D)
### Hosting Capacity & Violations
*   `ev_violations`: The number of EV-induced violations. It is calculated as the sum of voltage (<0.95 or >1.05 pu) and thermal (>100% loading) violations across the evaluated steps, **minus** the pre-existing grid violations from the No-EV reference run.
*   `losses`: Total line and transformer active power losses (MW) summed over the steps.
*   `grid_import`: Total active power imported from the external grid (MW) summed over the steps.
*   `viol_step_range`: The min and max steps where violations occurred.
*   `unique_assets`: The count of unique assets that experienced violations.

In [ ]:
def aggregate_results(res_df, no_ev_v, no_ev_l, pen, mode):
    v_buses = set()
    l_lines = set()
    t_trafos = set()
    for b in res_df['v_viol_buses']: v_buses.update(b)
    for l in res_df['l_viol_lines']: l_lines.update(l)
    for t in res_df['t_viol_trafos']: t_trafos.update(t)
    
    worst_b = res_df.loc[res_df['v_min'].idxmin(), 'worst_bus']
    worst_l = res_df.loc[res_df['line_load_max'].idxmax(), 'worst_line']
    
    viol_steps = res_df[(res_df['v_violations'] > 0) | (res_df['loading_violations'] > 0)]['step'].tolist()
    step_range = f"{min(viol_steps)}-{max(viol_steps)}" if viol_steps else "None"
    ev_v = max(0, res_df['v_violations'].sum() - no_ev_v) + max(0, res_df['loading_violations'].sum() - no_ev_l)
    
    return {
        'penetration': pen, 'mode': mode,
        'v_min': res_df['v_min'].min(), 'line_load_max': res_df['line_load_max'].max(), 'trafo_load_max': res_df['trafo_load_max'].max(),
        'losses_mw_sum': res_df['losses_mw'].sum(), 'grid_import_mw_sum': res_df['ext_grid_mw'].sum(),
        'ev_violations': ev_v, 'unique_v_buses': len(v_buses), 'unique_l_lines': len(l_lines), 'unique_t_trafos': len(t_trafos),
        'worst_bus': worst_b, 'worst_line': worst_l, 'viol_step_range': step_range
    }

if RUN_HOSTING_CAPACITY and not ev_df.empty:
    print("\n--- NO-EV REFERENCE (96 Steps) ---")
    no_ev_res = run_ac_power_flow(net, None, load_abs, renew_abs, list(range(96)))
    no_ev_v = no_ev_res['v_violations'].sum()
    no_ev_l = no_ev_res['loading_violations'].sum()
    print(f"Pre-existing Grid Violations: {no_ev_v} voltage, {no_ev_l} thermal")
    no_ev_agg = aggregate_results(no_ev_res, 0, 0, 0, 'No-EV')
    display(pd.DataFrame([no_ev_agg]))
    
    hc_results = []
    uncontrolled_hc = 0.0
    ga_hc = 0.0
    uncontrolled_violates = False
    ga_violates = False
    last_pass_pen_u = 0.0
    last_pass_pen_ga = 0.0
    
    print(f"\nSweeping penetrations: {PENETRATION_SWEEP}")
    for pen in PENETRATION_SWEEP:
        print(f"\nEvaluating Penetration: {pen*100:.0f}%")
        active_evs = generate_active_evs(pen)
        
        # Uncontrolled
        if not uncontrolled_violates:
            res_u = run_ac_power_flow(net, active_evs, load_abs, renew_abs, HC_STEPS)
            agg_u = aggregate_results(res_u, no_ev_v, no_ev_l, pen, 'uncontrolled')
            hc_results.append(agg_u)
            if agg_u['ev_violations'] > 0:
                uncontrolled_violates = True
                print(f"  -> Uncontrolled violates at {pen*100}%!")
            else:
                last_pass_pen_u = pen
                
        # GA Optimized
        if not ga_violates:
            best_active_evs = run_ga(active_evs)
            res_ga = run_ac_power_flow(net, best_active_evs, load_abs, renew_abs, HC_STEPS)
            agg_ga = aggregate_results(res_ga, no_ev_v, no_ev_l, pen, 'ga_optimized')
            hc_results.append(agg_ga)
            if agg_ga['ev_violations'] > 0:
                ga_violates = True
                print(f"  -> GA violates at {pen*100}%!")
            else:
                last_pass_pen_ga = pen
                
        if uncontrolled_violates and ga_violates:
            print("Both modes have reached limits in coarse sweep.")
            break
            
    # Bisection for Uncontrolled
    if uncontrolled_violates:
        print(f"\nFine-tuning Uncontrolled HC between {last_pass_pen_u*100}% and {(last_pass_pen_u+0.1)*100}%")
        for fine_pen in np.arange(last_pass_pen_u + 0.02, last_pass_pen_u + 0.1, 0.02):
            active_evs = generate_active_evs(fine_pen)
            res_u = run_ac_power_flow(net, active_evs, load_abs, renew_abs, HC_STEPS)
            agg_u = aggregate_results(res_u, no_ev_v, no_ev_l, fine_pen, 'uncontrolled')
            hc_results.append(agg_u)
            if agg_u['ev_violations'] > 0:
                break
            last_pass_pen_u = fine_pen
    else:
        last_pass_pen_u = PENETRATION_SWEEP[-1]
        
    # Bisection for GA
    if ga_violates:
        print(f"\nFine-tuning GA HC between {last_pass_pen_ga*100}% and {(last_pass_pen_ga+0.1)*100}%")
        for fine_pen in np.arange(last_pass_pen_ga + 0.02, last_pass_pen_ga + 0.1, 0.02):
            active_evs = generate_active_evs(fine_pen)
            best_active_evs = run_ga(active_evs)
            res_ga = run_ac_power_flow(net, best_active_evs, load_abs, renew_abs, HC_STEPS)
            agg_ga = aggregate_results(res_ga, no_ev_v, no_ev_l, fine_pen, 'ga_optimized')
            hc_results.append(agg_ga)
            if agg_ga['ev_violations'] > 0:
                break
            last_pass_pen_ga = fine_pen
    else:
        last_pass_pen_ga = PENETRATION_SWEEP[-1]

    print(f"\n--- FINAL EXACT HOSTING CAPACITY ---")
    print(f"Uncontrolled HC: {last_pass_pen_u*100:.0f}%")
    print(f"GA Optimized HC: {last_pass_pen_ga*100:.0f}%")
    print(f"Improvement: {(last_pass_pen_ga - last_pass_pen_u)*100:.0f}%")
    
    hc_df = pd.DataFrame(hc_results)
    hc_df = hc_df.sort_values(by=['mode', 'penetration'])
    hc_df.to_csv(HC_OUTPUT_PATH, index=False)
    print(f"Saved detailed HC results to {HC_OUTPUT_PATH}")

### 3.1 Rebound Spike Plot (20% Case)

In [ ]:
if RUN_HOSTING_CAPACITY and not ev_df.empty:
    active_evs_20 = generate_active_evs(0.20)
    best_active_evs_20 = run_ga(active_evs_20)
    res_u_96 = run_ac_power_flow(net, active_evs_20, load_abs, renew_abs, list(range(96)))
    res_ga_96 = run_ac_power_flow(net, best_active_evs_20, load_abs, renew_abs, list(range(96)))
    
    plt.figure(figsize=(12, 5))
    plt.plot(res_u_96['step'], res_u_96['ext_grid_mw'], label='Uncontrolled (20%)', color='red')
    plt.plot(res_ga_96['step'], res_ga_96['ext_grid_mw'], label='GA Optimized (20%)', color='green')
    plt.axvline(84, color='purple', linestyle='--', label='Price Drop (21:00)')
    plt.title('Grid Import MW over 24 Hours (Checking for Rebound Spikes)')
    plt.xlabel('Time Step (15-min intervals)')
    plt.ylabel('Grid Import (MW)')
    plt.legend()
    plt.grid(True)
    plt.show()

# 4. STAGE E: INTEGRATION EXPORT

In [ ]:
if not ev_df.empty and RUN_HOSTING_CAPACITY:
    try:
        best_active_evs_20
        melted_records = []
        base_date = pd.to_datetime("2021-01-01 00:00:00")
        for step in range(STEPS_PER_DAY):
            ts = base_date + pd.Timedelta(minutes=step * TIME_RESOLUTION_MINUTES)
            ts_str = ts.strftime("%Y-%m-%d %H:%M:%S")
            for idx, row in best_active_evs_20.iterrows():
                melted_records.append({
                    'timestamp': ts_str,
                    'bus': row['assigned_bus'],
                    'ev_load_mw': row[f'step_{step}'] / 1000.0,
                    'ev_load_uncontrolled_mw': active_evs_20.at[idx, f'step_{step}'] / 1000.0,
                    'ev_limit': 100,
                    'scenario_name': "GA_OPT_MEDIUM"
                })
        export_df = pd.DataFrame(melted_records)
        export_df.to_csv(OUTPUT_PATH, index=False)
        print(f"Exported 20% optimized profiles to {OUTPUT_PATH}")
    except NameError:
        print("No optimized EVs to export.")